#IS 470 Lab 6: Generalization and Overfitting

---

This data set contains information of cars purchased at the Auction.
<br>
We will use this file to predict the quality of buying decisions and visualize decision processes.
<br>
<br>
VARIABLE DESCRIPTIONS:<br>
Auction: Auction provider at which the  vehicle was purchased<br>
Color: Vehicle Color<br>
IsBadBuy: Identifies if the kicked vehicle was an avoidable purchase<br>
MMRCurrentAuctionAveragePrice: Acquisition price for this vehicle in average condition as of current day<br>
Size: The size category of the vehicle (Compact, SUV, etc.)<br>
TopThreeAmericanName:Identifies if the manufacturer is one of the top three American manufacturers<br>
VehBCost: Acquisition cost paid for the vehicle at time of purchase<br>
VehicleAge: The Years elapsed since the manufacturer's year<br>
VehOdo: The vehicles odometer reading<br>
WarrantyCost: Warranty price (term=36month  and millage=36K)<br>
WheelType: The vehicle wheel type description (Alloy, Covers)<br>
<br>
Target variable: **IsBadBuy**

In [1]:
# Mounting Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import classification_report
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn import preprocessing

## 1.Upload and clean data

In [ ]:
# Read data
car_kick = pd.read_csv("/content/drive/MyDrive/IS470_data/car_kick.csv")
car_kick

In [4]:
# Select the desired columns only
desired_columns = ['Auction', 'Color', 'IsBadBuy', 'MMRCurrentAuctionAveragePrice', 'Size','TopThreeAmericanName',
'VehBCost', 'VehicleAge', 'VehOdo', 'WarrantyCost', 'WheelType']
car_kick_desired = car_kick [desired_columns]

In [5]:
# Replacing 1 with Yes and 0 with No in the target column IsBadBuy
carAuction = car_kick_desired.copy() #why?
carAuction.loc[:, 'IsBadBuy'] = carAuction['IsBadBuy'].replace({0: 'No', 1: 'Yes'})

In [ ]:
# Examine variable type
carAuction.dtypes

In [6]:
# Change categorical variables to "category"
carAuction['Auction'] = carAuction['Auction'].astype('category')
carAuction['Color'] = carAuction['Color'].astype('category')
carAuction['IsBadBuy'] = carAuction['IsBadBuy'].astype('category')
carAuction['Size'] = carAuction['Size'].astype('category')
carAuction['TopThreeAmericanName'] = carAuction['TopThreeAmericanName'].astype('category')
carAuction['WheelType'] = carAuction['WheelType'].astype('category')

In [ ]:
# Create dummy variables
carAuction = pd.get_dummies(carAuction, columns=['Auction','Color','Size','TopThreeAmericanName','WheelType'], drop_first=True)
carAuction

In [ ]:
# Take the target and examine the porportion of target variable for each class
target = carAuction['IsBadBuy']
print(target.value_counts(normalize=True))

In [9]:
# Drop the target variable and put all the predictors in a new dataframe
predictors = carAuction.drop(['IsBadBuy'],axis=1)

In [ ]:
# Apply minmax normalization on predictors
min_max_scaler = preprocessing.MinMaxScaler()
predictors_normalized = pd.DataFrame(min_max_scaler.fit_transform(predictors))
predictors_normalized.columns = predictors.columns
predictors_normalized

## 2.Partition and balance the data set

In [ ]:
# Partition the data
predictors_train, predictors_test, target_train, target_test = train_test_split(predictors_normalized, target, test_size=0.1, random_state=0)
print(predictors_train.shape, predictors_test.shape, target_train.shape, target_test.shape)

In [12]:
# Taking steps to balance the train data
# Combine predictors_train and target_train into a single DataFrame
combined_train_df = pd.concat([predictors_train, target_train], axis=1)

# Separate majority and minority classes
majority_df = combined_train_df[combined_train_df['IsBadBuy'] == 'No']
minority_df = combined_train_df[combined_train_df['IsBadBuy'] == 'Yes']

# Undersample the majority class randomly
undersampled_majority = majority_df.sample(n=len(minority_df), random_state=5)

# Combine the undersampled majority class and the minority class
undersampled_data = pd.concat([undersampled_majority, minority_df])

# Shuffle the combined DataFrame to ensure randomness
balanced_data = undersampled_data.sample(frac=1, random_state=5)

# Split the balanced_data into predictors_train and target_train
predictors_train = balanced_data.drop(columns=['IsBadBuy'])
target_train = balanced_data['IsBadBuy']

In [ ]:
# Examine the porportion of target variable for training data set
print(target_train.value_counts(normalize=True))

In [ ]:
# Examine the porportion of target variable for testing data set
print(target_test.value_counts(normalize=True))

## 3.Generalization and Overfitting

### Build a decision tree model with max_depth = 2

In [ ]:
# Build a decision tree model on training data with max_depth = 2
model_tree1 = DecisionTreeClassifier(criterion = "entropy",random_state = 1, max_depth = 2)
model_tree1.fit(predictors_train, target_train)

In [ ]:
# Plot the tree
fig = plt.figure(figsize=(30,20))
tree.plot_tree(model_tree1,
               feature_names=list(carAuction.columns)[1:],
               class_names=['No','Yes'],
               filled=True)

In [17]:
# Make predictions on training and testing data
prediction_on_train = model_tree1.predict(predictors_train)
prediction_on_test = model_tree1.predict(predictors_test)

In [ ]:
# Examine the evaluation results on training data: accuracy, precision, recall, and f1-score
print(classification_report(target_train, prediction_on_train))

In [ ]:
# Examine the evaluation results on testing data: accuracy, precision, recall, and f1-score
print(classification_report(target_test, prediction_on_test))

### Build a decision tree model with max_depth = 5

In [ ]:
# Build a decision tree model on training data with max_depth = 5
model_tree2 = DecisionTreeClassifier(criterion = "entropy",random_state = 1, max_depth = 5)
model_tree2.fit(predictors_train, target_train)

In [22]:
# Make predictions on training and testing data
prediction_on_train = model_tree2.predict(predictors_train)
prediction_on_test = model_tree2.predict(predictors_test)

In [ ]:
# Examine the evaluation results on training data: accuracy, precision, recall, and f1-score
print(classification_report(target_train, prediction_on_train))

In [ ]:
# Examine the evaluation results on testing data: accuracy, precision, recall, and f1-score
print(classification_report(target_test, prediction_on_test))

### Build a decision tree model with max_depth = 30

In [ ]:
# Build a decision tree model on training data with max_depth = 30
model_tree3 = DecisionTreeClassifier(criterion = "entropy", random_state = 1, max_depth = 30)
model_tree3.fit(predictors_train, target_train)

In [26]:
# Make predictions on training and testing data
prediction_on_train = model_tree3.predict(predictors_train)
prediction_on_test = model_tree3.predict(predictors_test)

In [ ]:
# Examine the evaluation results on training data: accuracy, precision, recall, and f1-score
print(classification_report(target_train, prediction_on_train))

In [ ]:
# Examine the evaluation results on testing data: accuracy, precision, recall, and f1-score
print(classification_report(target_test, prediction_on_test))

Q1. Which decision tree model has the best overall performance (based on f1-scores and overal accuracy)?<br>

Q2. Describe the performance of the decision tree with max_depth = 30 on the train set. Despite almost perfect performance on the train set, why it is not working well on the test set?

Q3. What is your take on under-fiting, overfitting, and generalization? Which model underfits, which one overfit? Explain.

In [79]:
!jupyter nbconvert --to html "/content/drive/MyDrive/Colab Notebooks/IS470_lab06.ipynb"

[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/IS470_lab06.ipynb to html
[NbConvertApp] Writing 643959 bytes to /content/drive/MyDrive/Colab Notebooks/IS470_lab06.html
